In [34]:
%cd /home/brimmann/works/xRAG

/home/brimmann/works/xRAG


/home/brimmann/works/xRAG/.venv/lib/python3.9/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [26]:
from datasets import load_dataset, load_from_disk

In [25]:
# dataset_args = {}
data_files = {"all_data": "/home/brimmann/works/atlas/data/corpora/wiki/enwiki-dec2021/text-list-100-sec.jsonl"}

In [29]:
native_dataset_file = "data/all_data/"

In [30]:
raw_data = load_from_disk(native_dataset_file)

Loading dataset from disk:   0%|          | 0/40 [00:00<?, ?it/s]

In [45]:
sample_raw_data = raw_data.select(range(10))

In [47]:
train_dataset = sample_raw_data.select(range(7))
dev_dataset = sample_raw_data.select(range(7, 10))

In [48]:
train_dataset.to_json("train.jsonl", orient="records", lines=True)
dev_dataset.to_json("dev.jsonl", orient="records", lines=True)

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

1497

In [49]:
    import os
    print(os.getcwd())

/home/brimmann/works/xRAG


In [35]:
from src.language_modeling.preprocessing import encode_with_chat_format_pretrain
from functools import partial

In [ ]:
from transformers import AutoTokenizer
model_name = "google/gemma-2-2b"
tokenizer = AutoTokenizer.from_pretrained(model_name)
chat_format = "mistral"
max_seq_length = 336

In [37]:
partial_func_pretrain = partial(
    encode_with_chat_format_pretrain,
    tokenizer=tokenizer,
    max_seq_length=max_seq_length,
    retrieval_embed_length=1,
    chat_format="mistral"
    )

In [40]:
sample_raw_data

Dataset({
    features: ['id', 'title', 'section', 'text'],
    num_rows: 5
})

In [43]:
lm_dataset_sample = sample_raw_data.map(
    partial_func_pretrain,
    remove_columns=[name for name in sample_raw_data.column_names if name not in ["input_ids", "labels", "attention_mask"]],
)

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [44]:
lm_dataset_sample

Dataset({
    features: ['xrag_input_ids', 'xrag_labels', 'retriever_input_text'],
    num_rows: 5
})

In [41]:
lm_dataset_sample.set_format("pt")

In [11]:
prompt = "Background: {xrag_token}, which also means:"

prompt.format_map(dict(xrag_token="xrag-token", document="capital of france is paris"))

'Background: xrag-token, which also means:'

In [18]:
from src.language_modeling.preprocessing import _encode_chat_format

In [ ]:
instruction = "Background: <xRAG>, which also means:"
document = "Against the Grain (Kurupt album) |  Against the Grain is the fourth studio album by American rapper Kurupt and his first on Death Row Records as a solo artist. Kurupt signed back onto Death Row Records, except as a solo artist in 2002. The album was delay from its planned 2004 release and was released in August 2005. It was Death Row's first freshly recorded album in over four years. The album went almost unnoticed due to the lack of promotion by Koch Records, which distributes all of Death Row's albums. It was the final original album released by the label."
messages = [
    {"role":"user","content":instruction},
    {"role":"assistant","content":document},
]

In [19]:
r = _encode_chat_format(messages,tokenizer,max_seq_length,chat_format)

In [20]:
r

{'input_ids': tensor([     2, 235309,  19647, 235307,  25493, 235292,    968, 235297, 190757,
          14514,    948,   1170,   3454, 235292, 113701,  19647, 235307,  83993,
            573,  64332,    591,  37280,  28218,   8022, 235275,   1420,    139,
          83993,    573,  64332,    603,    573,  12201,  11557,   8022,    731,
           3725,  57963,  18321,  28218,    578,    926,   1370,    611,  14958,
          14923,  17753,    685,    476,   7212,   9161, 235265,  18321,  28218,
          10795,   1355,  10401,  14958,  14923,  17753, 235269,   7694,    685,
            476,   7212,   9161,    575, 235248, 235284, 235276, 235276, 235284,
         235265,    714,   8022,    729,  10509,    774,   1277,  14196, 235248,
         235284, 235276, 235276, 235310,   4236,    578,    729,   8417,    575,
           4826, 235248, 235284, 235276, 235276, 235308, 235265,   1165,    729,
          14958,  14923, 235303, 235256,   1370,  51644,  11659,   8022,    575,
           1163